# Colab GRU Training — Observatorio de Opiniones E-commerce
**Proyecto Final PLN · Maestría en IA · UCB San Pablo**

Capa orquestadora de entrenamiento con GPU. El código fuente vive en PyCharm local.

**Parámetros mejorados vs CPU local:**
- max_vocab_size: 10.000 → 20.000
- max_length: 35 → 72 (percentil 95 real del corpus)
- GPU T4 en lugar de CPU

## CELDA 0 — Instalación de dependencias
Instala TF 2.18 y tf-keras que provee la API de Keras 2 (Tokenizer, pad_sequences).
tf-keras es el paquete oficial de compatibilidad hacia atrás de Keras 2 para TF 2.16+.
Reinicia el runtime después de ejecutar esta celda.

In [ ]:
# Instala tf-keras: provee Tokenizer y pad_sequences compatibles con TF 2.18
!pip install tf-keras==2.18.* -q

# Instala el resto de dependencias del proyecto
!pip install scikit-learn pandas joblib datasets nltk -q

print('Instalacion completada — reinicia el runtime antes de continuar')

## CELDA 0b — Descarga y filtrado de embeddings FastText en español
Descarga los vectores FastText crawl para español y filtra solo las palabras del vocabulario del tokenizador.
Esto reduce drásticamente la memoria y el tiempo de carga en Colab.

In [ ]:
# Descarga el archivo de embeddings FastText en espanol desde Facebook AI
# cc.es.300.vec.gz: 2 millones de palabras en espanol, dimension 300
# Solo cargamos las palabras del vocabulario del tokenizador — no los 2M completos
!wget -q https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.es.300.vec.gz
!gunzip -f cc.es.300.vec.gz
print('Embeddings FastText descargados y descomprimidos')
print('Archivo: cc.es.300.vec')

## CELDA 1 — Reinicio del runtime
Obligatorio después de instalar tf-keras.
Ejecuta esta celda y luego continúa desde la CELDA 2.

In [ ]:
# Reinicia el runtime para aplicar los cambios de instalación
import os
os.kill(os.getpid(), 9)

## CELDA 2 — Verificación de entorno
Confirma TF, tf-keras y GPU antes de continuar.

In [ ]:
import tensorflow as tf                      # framework de deep learning
import tf_keras as keras                     # API Keras 2 compatible con TF 2.18
import numpy as np                           # operaciones numéricas

# Verifica versiones
print('TF version:    ', tf.__version__)     # debe mostrar 2.18.x
print('Keras version: ', keras.__version__)  # debe mostrar 2.18.x

# Verifica GPU
gpus = tf.config.list_physical_devices('GPU')
print('GPUs disponibles:', gpus)

if not gpus:
    print('ADVERTENCIA: No se detecto GPU — ve a Runtime > Change runtime type > T4 GPU')
else:
    print('GPU detectada correctamente — el entrenamiento sera rapido')

## CELDA 3 — Montar Google Drive y cargar CSV
Monta Drive y carga el CSV procesado generado por el pipeline local.
El CSV ya está en: Mi unidad/Maestria Ciencia de Datos/PROC_LENG_NATURAL/PRO_FINAL/COLAB/

In [ ]:
from google.colab import drive               # módulo para montar Google Drive
import pandas as pd                          # manipulación de tablas de datos

# Monta Google Drive en /content/drive
drive.mount('/content/drive')

# Ruta base del proyecto en Drive
BASE_DRIVE = '/content/drive/MyDrive/Maestria Ciencia de Datos/PROC_LENG_NATURAL/PRO_FINAL/COLAB'

# Carga el CSV balanceado generado por el pipeline local
CSV_PATH = f'{BASE_DRIVE}/reviews_es_balanced.csv'
df = pd.read_csv(CSV_PATH)                   # lee el CSV con 11.998 registros
print('CSV cargado:', df.shape)
print(df['sentiment_label'].value_counts())

## CELDA 4 — Configuración de parámetros
Parámetros mejorados para GPU — superiores a los del entrenamiento CPU local.
Semilla fija para reproducibilidad total.

In [ ]:
import random

# Semilla fija — igual que en el proyecto local
RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)             # semilla para TF
np.random.seed(RANDOM_STATE)                 # semilla para NumPy
random.seed(RANDOM_STATE)                    # semilla para Python random

# Parámetros mejorados para GPU
MAX_VOCAB_SIZE = 20000   # vocabulario completo (vs 10000 en CPU)
EMBEDDING_DIM  = 100     # dimensión de embeddings
GRU_UNITS      = 64      # unidades GRU
DROPOUT        = 0.30    # dropout para evitar sobreajuste
MAX_LENGTH     = 72      # percentil 95 real del corpus (vs 35 en CPU)
EPOCHS         = 20      # máximo de épocas
BATCH_SIZE     = 64      # ejemplos por paso
PATIENCE       = 3       # épocas sin mejora antes de detener

# Orden contractual de clases — INMUTABLE
LABEL_ID_TO_NAME = {0: 'NEGATIVO', 1: 'NEUTRO', 2: 'POSITIVO'}

print('Parametros configurados:')
print(f'  MAX_VOCAB_SIZE: {MAX_VOCAB_SIZE}')
print(f'  MAX_LENGTH:     {MAX_LENGTH}')
print(f'  GRU_UNITS:      {GRU_UNITS}')
print(f'  DROPOUT:        {DROPOUT}')

## CELDA 5 — Preprocesamiento y tokenización
El tokenizador se ajusta SOLO con train — regla anti-fuga de datos.
Nunca se llama fit() sobre validation o test.

In [ ]:
# Importa desde tf_keras — API Keras 2 compatible con TF 2.18
from tf_keras.preprocessing.text import Tokenizer           # convierte texto a secuencias numéricas
from tf_keras.preprocessing.sequence import pad_sequences   # iguala longitud de secuencias

# Separa los splits del CSV
train_df = df[df['split'] == 'train'].copy()       # 9000 registros para entrenar
val_df   = df[df['split'] == 'validation'].copy()  # 1500 registros para validar
test_df  = df[df['split'] == 'test'].copy()        # 998 registros para evaluar
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

# Preprocesamiento mínimo — el tokenizador maneja la representación
train_texts = train_df['text_raw'].str.lower().str.strip().tolist()
val_texts   = val_df['text_raw'].str.lower().str.strip().tolist()
test_texts  = test_df['text_raw'].str.lower().str.strip().tolist()

# Tokenizador: aprende vocabulario SOLO desde train
tokenizer = Tokenizer(
    num_words=MAX_VOCAB_SIZE,    # limita a las 20000 palabras más frecuentes
    oov_token='<OOV>'           # token para palabras fuera del vocabulario
)
tokenizer.fit_on_texts(train_texts)   # ajusta SOLO con train
print(f'Vocabulario aprendido: {len(tokenizer.word_index)} palabras unicas')

# Convierte textos a secuencias numéricas de longitud fija
def textos_a_secuencias(textos):
    seqs = tokenizer.texts_to_sequences(textos)              # IDs numéricos
    return pad_sequences(seqs, maxlen=MAX_LENGTH,            # padding fijo a 72 tokens
                         padding='post', truncating='post')

X_train = textos_a_secuencias(train_texts)   # matriz (9000, 72)
X_val   = textos_a_secuencias(val_texts)     # matriz (1500, 72)
X_test  = textos_a_secuencias(test_texts)    # matriz (998, 72)

# Etiquetas como arrays numpy — requerido por Keras
y_train = np.array(train_df['sentiment_id'].tolist())
y_val   = np.array(val_df['sentiment_id'].tolist())
y_test  = np.array(test_df['sentiment_id'].tolist())

print(f'X_train shape: {X_train.shape} | y_train shape: {y_train.shape}')

## CELDA 6 — Construcción y entrenamiento de la GRU
Arquitectura contractual: Embedding → GRU → Dense(softmax).
EarlyStopping detiene el entrenamiento si val_loss no mejora en 3 épocas.
restore_best_weights recupera los pesos de la mejor época.

In [ ]:
from tf_keras.models import Sequential
from tf_keras.layers import Embedding, GRU, Dense, SpatialDropout1D
from tf_keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tf_keras.optimizers import Adam
import numpy as np
import io
import time

# PASO 1: Carga los embeddings FastText filtrando solo el vocabulario del tokenizador
# Esto evita cargar 2 millones de vectores — solo cargamos los que necesitamos
print('Cargando embeddings FastText en espanol...')
EMBEDDING_DIM_FT = 300                       # FastText usa dimension 300

# Vocabulario del tokenizador: mapa palabra -> ID numerico
word_index = tokenizer.word_index            # diccionario {palabra: id}

# Matriz de embeddings inicializada en ceros — shape (vocab+1, 300)
embedding_matrix = np.zeros((MAX_VOCAB_SIZE + 1, EMBEDDING_DIM_FT))

encontradas = 0                              # palabras del vocab encontradas en FastText
with io.open('cc.es.300.vec', 'r', encoding='utf-8', newline='
', errors='ignore') as f:
    n, d = map(int, f.readline().split())    # primera linea: n_palabras dimension
    for line in f:
        tokens = line.rstrip().split(' ')
        word = tokens[0]                     # primera columna: la palabra
        if word in word_index:               # solo si esta en nuestro vocabulario
            idx = word_index[word]           # ID numerico de esa palabra
            if idx < MAX_VOCAB_SIZE + 1:     # dentro del limite del vocabulario
                vector = np.array(tokens[1:], dtype='float32')  # vector de 300 dimensiones
                embedding_matrix[idx] = vector  # inserta el vector en la matriz
                encontradas += 1

cobertura = encontradas / min(len(word_index), MAX_VOCAB_SIZE) * 100
print(f'Palabras encontradas en FastText: {encontradas} / {min(len(word_index), MAX_VOCAB_SIZE)}')
print(f'Cobertura del vocabulario: {cobertura:.1f}%')
print(f'Matriz de embeddings: {embedding_matrix.shape}')

# PASO 2: Construye la GRU con embeddings preentrenados
# trainable=False: los embeddings se mantienen fijos — la GRU aprende solo la clasificacion
model = Sequential([
    Embedding(
        input_dim=MAX_VOCAB_SIZE + 1,        # tamano del vocabulario
        output_dim=EMBEDDING_DIM_FT,         # dimension 300 de FastText
        weights=[embedding_matrix],          # pesos iniciales = embeddings preentrenados
        input_length=MAX_LENGTH,             # longitud fija: 72 palabras
        trainable=False,                     # congela embeddings — no se actualizan
    ),
    SpatialDropout1D(0.2),                   # dropout espacial compatible con cuDNN
    GRU(
        units=GRU_UNITS,                     # 64 unidades de memoria recurrente
        dropout=DROPOUT,                     # 30% dropout sin recurrent_dropout
    ),
    Dense(3, activation='softmax'),          # 3 clases: NEGATIVO / NEUTRO / POSITIVO
]),

model.compile(
    optimizer=Adam(learning_rate=0.001),     # Adam estandar
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=0.0001,
)

print('Iniciando entrenamiento con embeddings preentrenados FastText...')
inicio = time.time()
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=64,
    callbacks=[early_stopping, reduce_lr],
    verbose=1,
)
duracion = time.time() - inicio
print(f'Entrenamiento completado en {duracion:.1f} segundos')
print(f'Epocas ejecutadas: {len(history.history["loss"])}')

## CELDA 7 — Evaluación sobre el test
Mismo test que usó el pipeline local — comparación justa con Naive Bayes.
Genera todas las métricas obligatorias del contrato §9.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)
import json

# Predicción sobre test
inicio_inf = time.time()
probs = model.predict(X_test, verbose=0)             # probabilidades shape (998, 3)
tiempo_inf = (time.time() - inicio_inf) / len(y_test)
predictions = np.argmax(probs, axis=1)               # clase con mayor probabilidad

# Métricas obligatorias del contrato §9
gru_metrics = {
    'model':           'GRU',
    'accuracy':        round(float(accuracy_score(y_test, predictions)), 4),
    'precision_macro': round(float(precision_score(y_test, predictions, average='macro')), 4),
    'recall_macro':    round(float(recall_score(y_test, predictions, average='macro')), 4),
    'f1_macro':        round(float(f1_score(y_test, predictions, average='macro')), 4),
    'f1_weighted':     round(float(f1_score(y_test, predictions, average='weighted')), 4),
    'f1_per_class':    {
        LABEL_ID_TO_NAME[i]: round(float(v), 4)
        for i, v in enumerate(f1_score(y_test, predictions, average=None))
    },
    'inference_time_s': round(tiempo_inf, 6),
    'confusion_matrix': confusion_matrix(y_test, predictions).tolist(),
    'training_epochs':  len(history.history['loss']),
    'max_vocab_size':   MAX_VOCAB_SIZE,
    'max_length':       MAX_LENGTH,
    'device':           'GPU Colab T4',
}

print('GRU GPU — Resultados:')
print(f'  Accuracy:  {gru_metrics["accuracy"]}')
print(f'  F1 macro:  {gru_metrics["f1_macro"]}')
print(f'  Epocas:    {gru_metrics["training_epochs"]}')

## CELDA 8 — Generación de gráficos
Curvas de entrenamiento y matriz de confusión como PNG.
Sobreescriben los generados por el entrenamiento CPU local.

In [ ]:
import matplotlib
matplotlib.use('Agg')                                # backend sin pantalla
import matplotlib.pyplot as plt

CLASS_NAMES = ['NEGATIVO', 'NEUTRO', 'POSITIVO']     # orden contractual

# Curvas de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['accuracy'],     label='Train accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation accuracy')
axes[0].set_title('GRU GPU — Accuracy por epoca')
axes[0].set_xlabel('Epoca')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[1].plot(history.history['loss'],     label='Train loss')
axes[1].plot(history.history['val_loss'], label='Validation loss')
axes[1].set_title('GRU GPU — Loss por epoca')
axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('Loss')
axes[1].legend()
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.close()
print('Curvas guardadas')

# Matriz de confusión
cm = np.array(gru_metrics['confusion_matrix'])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, title, fmt in zip(
    axes,
    [cm, cm.astype(float) / cm.sum(axis=1, keepdims=True)],
    ['GRU GPU — Matriz absoluta', 'GRU GPU — Matriz normalizada'],
    ['d', '.2f']
):
    im = ax.imshow(data, cmap='Blues')
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel('Clase predicha')
    ax.set_ylabel('Clase real')
    ax.set_title(title)
    plt.colorbar(im, ax=ax)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, format(data[i, j], fmt), ha='center', va='center',
                    color='white' if data[i, j] > data.max() / 2 else 'black')
plt.tight_layout()
plt.savefig('/content/confusion_matrix_gru.png', dpi=150, bbox_inches='tight')
plt.close()
print('Matriz de confusion guardada')

## CELDA 9 — Guardar artefactos en Google Drive
Guarda modelo, tokenizador, métricas y gráficos en Drive.
Desde ahí los copias manualmente al proyecto local en PyCharm.

In [ ]:
import os
import shutil

# Carpeta de salida en Google Drive
OUTPUT_DIR = f'{BASE_DRIVE}/artifacts_gpu'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Guarda el modelo GRU entrenado con GPU en formato .keras
model.save(f'{OUTPUT_DIR}/gru_sentiment.keras')       # arquitectura + pesos
print('Modelo GRU guardado')

# Guarda el tokenizador con vocab=20000
with open(f'{OUTPUT_DIR}/tokenizer.json', 'w', encoding='utf-8') as f:
    f.write(tokenizer.to_json())                      # serializa tokenizador a JSON
print('Tokenizador guardado')

# Guarda las métricas de la GRU
with open(f'{OUTPUT_DIR}/gru_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(gru_metrics, f, indent=2, ensure_ascii=False)
print('Metricas GRU guardadas')

# Copia los gráficos generados
shutil.copy('/content/training_curves.png',     f'{OUTPUT_DIR}/training_curves.png')
shutil.copy('/content/confusion_matrix_gru.png', f'{OUTPUT_DIR}/confusion_matrix_gru.png')
print('Graficos copiados')

print(f'Todos los artefactos guardados en: {OUTPUT_DIR}')

## CELDA 10 — Actualizar champion.json
Compara F1 macro de NB local vs GRU GPU y determina el nuevo champion.
El archivo champion.json gobierna la interfaz Gradio — nunca se hardcodea.

In [ ]:
# Carga métricas del Naive Bayes entrenado localmente
NB_METRICS_PATH = f'{BASE_DRIVE}/naive_bayes_metrics.json'
with open(NB_METRICS_PATH, 'r', encoding='utf-8') as f:
    nb_metrics = json.load(f)

# Determina el champion por F1 macro — métrica principal del contrato §9
f1_nb  = nb_metrics['f1_macro']
f1_gru = gru_metrics['f1_macro']
champion = 'Naive Bayes' if f1_nb >= f1_gru else 'GRU'

# Genera champion.json — archivo que gobierna la interfaz Gradio
champion_data = {
    'champion':     champion,
    'f1_macro_nb':  f1_nb,
    'f1_macro_gru': f1_gru,
    'nb_device':    'CPU local',
    'gru_device':   'GPU Colab T4',
}
with open(f'{OUTPUT_DIR}/champion.json', 'w', encoding='utf-8') as f:
    json.dump(champion_data, f, indent=2, ensure_ascii=False)

# Genera comparison.json — tabla comparativa completa
comparison = {'Naive Bayes': nb_metrics, 'GRU': gru_metrics}
with open(f'{OUTPUT_DIR}/comparison.json', 'w', encoding='utf-8') as f:
    json.dump(comparison, f, indent=2, ensure_ascii=False)

print('RESULTADO FINAL:')
print(f'  F1 macro Naive Bayes (CPU): {f1_nb}')
print(f'  F1 macro GRU (GPU):         {f1_gru}')
print(f'  Champion: {champion}')
print(f'champion.json guardado en {OUTPUT_DIR}')

## CELDA 11 — Instrucciones de integración al proyecto local
Pasos para copiar los artefactos GPU al proyecto PyCharm.

In [ ]:
print('INSTRUCCIONES DE INTEGRACION AL PROYECTO LOCAL:')
print('================================================')
print()
print('Los artefactos ya estan en Google Drive en:')
print(f'  {OUTPUT_DIR}')
print()
print('Como Drive esta sincronizado en G:\My Drive, copia desde:')
print('  G:\My Drive\Maestria Ciencia de Datos\PROC_LENG_NATURAL\PRO_FINAL\COLAB\artifacts_gpu\')
print()
print('Archivos a copiar al proyecto local:')
print('  gru_sentiment.keras      -> artifacts/models/')
print('  tokenizer.json           -> artifacts/tokenizer/')
print('  gru_metrics.json         -> artifacts/metrics/')
print('  champion.json            -> artifacts/metrics/')
print('  comparison.json          -> artifacts/metrics/')
print('  training_curves.png      -> artifacts/figures/')
print('  confusion_matrix_gru.png -> artifacts/figures/')
print()
print('Luego reinicia Gradio: python src/app/gradio_app.py')